# Sepsis Early-Warning System — Live Demo App
Runs the trained sepsis models from `sepsis_v4.ipynb` (LightGBM ensemble + reliability-gated GRU-D + decision layer) inside a Gradio web app. The separate DenseNet-121 X-ray branch is optional and is not generated by default.

**How to run (interactive, not Commit):**
1. *+ Add Input* → **Notebooks → Your Work** → add the output of your **sepsis FINAL** notebook (it contains every model,
   including the X-ray model), plus the **Prediction of Sepsis** and **NIH Chest X-rays** datasets.
2. Settings → **Internet ON** (GPU optional; CPU is fast enough).
3. **Run All**. The last cell prints a link like `https://xxxx.gradio.live`: open it in Chrome. Keep this Kaggle tab open while presenting.

Research prototype for clinical decision support. Not a diagnostic device.

In [ ]:
%pip install -q gradio==6.29.0
print('✓ step 1 of 9 finished')

In [ ]:
import os, sys, json, time, math, glob, random, warnings
import numpy as np, pandas as pd, torch, torch.nn as nn, torch.nn.functional as F
import lightgbm as lgb
import matplotlib; matplotlib.use('Agg')
from matplotlib.figure import Figure
warnings.filterwarnings('ignore')
INPUT_ROOT = os.environ.get('SEPSIS_INPUT_ROOT', '/kaggle/input')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

def walk_find(names, roots=(INPUT_ROOT, '/kaggle/working')):
    found = {}
    for root in roots:
        if not os.path.isdir(root): continue
        for dp, dns, fns in os.walk(root, followlinks=True):
            dns[:] = [d for d in dns if not d.lower().startswith('images')]
            for f in fns:
                if f in names and f not in found: found[f] = os.path.join(dp, f)
    return found

# X-ray images/metadata may be found in the attached authorized dataset. Model and
# metric artifacts are restricted to the selected bundle directory.
F_ = walk_find({'Data_Entry_2017.csv'})
_bundle_env = os.environ.get('SEPSIS_BUNDLE_PATH')
if _bundle_env:
    _bundle_path = os.path.abspath(os.path.expanduser(_bundle_env))
    if os.path.basename(_bundle_path) != 'sepsis_monitor_bundle.pt' or not os.path.isfile(_bundle_path):
        raise RuntimeError('SEPSIS_BUNDLE_PATH must point to an existing sepsis_monitor_bundle.pt file.')
else:
    _bundle_candidates = []
    for _root in (INPUT_ROOT, '/kaggle/working'):
        if not os.path.isdir(_root): continue
        for _dp, _dns, _fns in os.walk(_root, followlinks=True):
            _dns[:] = [d for d in _dns if not d.lower().startswith('images')]
            if 'sepsis_monitor_bundle.pt' in _fns:
                _bundle_candidates.append(os.path.abspath(os.path.join(_dp, 'sepsis_monitor_bundle.pt')))
    _bundle_candidates = sorted(set(os.path.normcase(q) for q in _bundle_candidates))
    if len(_bundle_candidates) != 1:
        _found = '\n'.join(_bundle_candidates) if _bundle_candidates else '(none found)'
        raise RuntimeError('Expected exactly one sepsis_monitor_bundle.pt candidate; found ' + str(len(_bundle_candidates)) +
                           ':\n' + _found + '\nSet SEPSIS_BUNDLE_PATH to the exact bundle file to select it.')
    _bundle_path = _bundle_candidates[0]
BUNDLE_DIR = os.path.dirname(_bundle_path)
# Bundles use pickle-backed torch serialization. Load only a trusted training output.
B = torch.load(_bundle_path, map_location='cpu', weights_only=False)
_required = {'cfg', 'policy', 'stack_weights', 'norm', 'variables', 'rgn_state_dict', 'lightgbm_files'}
if not isinstance(B, dict) or not _required.issubset(B):
    raise RuntimeError('Selected file is not a valid sepsis monitor bundle: required bundle keys are missing.')
_lgb_files = B['lightgbm_files']
if not isinstance(_lgb_files, (list, tuple)) or not _lgb_files or any(not isinstance(f, str) or os.path.basename(f) != f for f in _lgb_files):
    raise RuntimeError('Bundle lightgbm_files must be a non-empty list of companion filenames beside the bundle.')
_missing_lgb = [f for f in _lgb_files if not os.path.isfile(os.path.join(BUNDLE_DIR, f))]
if _missing_lgb:
    raise FileNotFoundError('Missing LightGBM companion files beside selected bundle: ' + ', '.join(_missing_lgb))
SPLIT_PATH = os.path.join(BUNDLE_DIR, 'split_patient_ids.json')
if not os.path.isfile(SPLIT_PATH):
    raise FileNotFoundError('Required held-out cohort file is missing: ' + SPLIT_PATH +
                            '. Run the training notebook and attach its results directory.')
with open(SPLIT_PATH, encoding='utf-8') as _f:
    SPLIT_IDS = json.load(_f)
if not isinstance(SPLIT_IDS, dict) or not isinstance(SPLIT_IDS.get('test'), list):
    raise ValueError('split_patient_ids.json must contain a test list of patient IDs.')
CFG = B['cfg']; POLICY = B['policy']; W_STACK = B['stack_weights']
NORM = {k: (np.array(v, dtype=np.float32) if isinstance(v, list) else v) for k, v in B['norm'].items()}
VITALS, LABS, DEMO = B['variables']['vitals'], B['variables']['labs'], B['variables']['demo']
DYN = VITALS + LABS; NV, NL = len(VITALS), len(LABS)
MC = int(CFG.get('mc_passes', 30))
print('models from:', BUNDLE_DIR, '| device:', device, '| alert threshold ? =', round(POLICY['tau'], 4))
print('✓ step 2 of 9 finished')

In [ ]:
# ---- exact feature + model code copied from the training notebook (so predictions match) ----
KEY_LABS = ['Lactate', 'WBC', 'Creatinine', 'Platelets', 'Bilirubin_total', 'BUN']

def lgb_feats_from(Xr, Mk, g, t, Dm):
    """Xr raw values, Mk observation mask, g patient codes, t hour index, Dm demographics -> feature table."""
    Xr = Xr.copy(); Xr[~Mk] = np.nan
    D = pd.DataFrame(Xr, columns=DYN)
    ff = D.groupby(g).ffill()
    obs_t = pd.DataFrame(np.where(Mk, t[:, None].astype(np.float32), np.nan), columns=DYN).groupby(g).ffill().values
    delta = np.where(np.isnan(obs_t), 999., t[:, None] - obs_t)
    fv = ff[VITALS]; gv = fv.groupby(g)
    sh = np.stack([fv.values] + [gv.shift(k).values for k in range(1, 6)], 0)
    with np.errstate(all='ignore'), warnings.catch_warnings():
        warnings.simplefilter('ignore')
        win = [np.nanmean(sh, 0), np.nanstd(sh, 0), np.nanmax(sh, 0) - np.nanmin(sh, 0), fv.values - sh[-1]]
    cmax = fv[['HR', 'Temp', 'Resp']].groupby(g).cummax().values
    cmin = fv[['MAP', 'SBP', 'O2Sat']].groupby(g).cummin().values
    kl = ff[KEY_LABS]; ltr = kl.values - kl.groupby(g).shift(24).values
    HR, SBP, Tp, RR, PaCO2, WBC = (ff[c].values for c in ['HR', 'SBP', 'Temp', 'Resp', 'PaCO2', 'WBC'])
    with np.errstate(all='ignore'):
        sirs = ((Tp > 38) | (Tp < 36)).astype(np.int8) + (HR > 90) + ((RR > 20) | (PaCO2 < 32)) + ((WBC > 12) | (WBC < 4))
        qsofa = (RR >= 22).astype(np.int8) + (SBP <= 100)
        shock = HR / SBP
    labs_seen = pd.DataFrame(Mk[:, NV:].astype(np.int8)).groupby(g).cummax().sum(1).values
    cols = ([ff.values, delta] + win + [cmax, cmin, ltr, np.c_[shock, sirs, qsofa, labs_seen], Dm])
    names = (DYN + [f'{v}_dt' for v in DYN] + [f'{v}_{s}6h' for s in ['mean', 'std', 'range', 'trend'] for v in VITALS]
             + ['HR_max', 'Temp_max', 'Resp_max', 'MAP_min', 'SBP_min', 'O2Sat_min'] + [f'{v}_trend24h' for v in KEY_LABS]
             + ['shock_index', 'sirs', 'qsofa', 'labs_seen'] + DEMO)
    return pd.DataFrame(np.hstack(cols).astype(np.float32), columns=names)

LOG48 = math.log1p(48.)

def featurize(x, m, c):
    B, T, D = x.shape
    t = torch.arange(T, device=x.device).view(1, T, 1).expand(B, T, D)
    last = torch.cummax(torch.where(m > 0, t, torch.zeros_like(t)), dim=1)[0]
    ever = torch.cummax(m, dim=1)[0]
    xf = torch.gather(x, 1, last) * ever
    delta = torch.where(ever > 0, (t - last).float(), torch.full_like(x, 48.)).clamp(max=48.)
    dn = torch.log1p(delta) / LOG48
    v, l = slice(0, NV), slice(NV, D)
    rel = torch.stack([m[..., v].mean(-1), 1 - dn[..., v].mean(-1), ever[..., v].mean(-1),
                       m[..., l].mean(-1), 1 - dn[..., l].mean(-1), ever[..., l].mean(-1)], -1)
    avail = torch.stack([(rel[..., 2] > 0).float(), (rel[..., 5] > 0).float(), torch.ones_like(rel[..., 0])], -1)
    return {'vit': torch.cat([xf[..., v], m[..., v], dn[..., v]], -1),
            'lab': torch.cat([xf[..., l], m[..., l], dn[..., l]], -1),
            'xf': xf, 'm': m, 'delta': delta, 'dn': dn,
            'ctx': c, 'rel': rel, 'avail': avail, 'E': 0.5 * rel[..., 1] + 0.5 * rel[..., 5]}


class ReliabilityGatedGRUD(nn.Module):
    def __init__(self, H=CFG['hidden'], p=CFG['dropout']):
        super().__init__(); self.p = p
        self.decay_w = nn.Parameter(torch.full((NV + NL,), 0.05)); self.decay_b = nn.Parameter(torch.zeros(NV + NL))
        self.vit_in = nn.Sequential(nn.Linear(3 * NV, H), nn.ReLU()); self.vit_gru = nn.GRU(H, H, batch_first=True)
        self.lab_in = nn.Sequential(nn.Linear(3 * NL, H), nn.ReLU()); self.lab_gru = nn.GRU(H, H, batch_first=True)
        self.ctx = nn.Sequential(nn.Linear(7, H), nn.ReLU(), nn.Linear(H, H), nn.ReLU())
        self.gate = nn.Sequential(nn.Linear(3 * H + 6, H), nn.ReLU(), nn.Linear(H, 3))
        nn.init.constant_(self.gate[-1].bias, 2.0)
        self.head = nn.Sequential(nn.Linear(3 * H + 6, H), nn.ReLU()); self.out = nn.Linear(H, 1)
    def encode(self, f):
        gamma = torch.exp(-F.relu(self.decay_w * f['delta'] + self.decay_b))          # GRU-D input decay
        xh = f['xf'] * (f['m'] + (1 - f['m']) * gamma)                               # decays to mean (=0 after z-score)
        v, l = slice(0, NV), slice(NV, NV + NL)
        hv, _ = self.vit_gru(self.vit_in(torch.cat([xh[..., v], f['m'][..., v], f['dn'][..., v]], -1)))
        hl, _ = self.lab_gru(self.lab_in(torch.cat([xh[..., l], f['m'][..., l], f['dn'][..., l]], -1)))
        hc = self.ctx(f['ctx'])
        w = torch.sigmoid(self.gate(torch.cat([hv, hl, hc, f['rel']], -1))) * f['avail']   # per-modality trust
        z = torch.cat([w[..., :1] * hv, w[..., 1:2] * hl, w[..., 2:] * hc, f['rel']], -1)
        return z, {'gates': w}
    def head_fwd(self, h, mc=False):
        on = self.training or mc
        return self.out(F.dropout(self.head(F.dropout(h, self.p, on)), self.p, on)).squeeze(-1)
    def forward(self, f, mc=False): return self.head_fwd(self.encode(f)[0], mc)


def to_model_inputs(Xr, Dm, norm):
    """Raw values (NaN = not measured) -> z-scored values Z, mask M, context C. Shared by training and the live monitor."""
    M = ~np.isnan(Xr)
    Z = np.where(M, (np.clip(Xr, norm['lo'], norm['hi']) - norm['mu']) / norm['sd'], 0.).astype(np.float32)
    age, gender, u1, u2, hadm, icu = [Dm[:, j] for j in range(6)]
    zc = lambda v, k: np.nan_to_num((np.clip(v, norm[k+'_lo'], norm[k+'_hi']) - norm[k+'_mu']) / norm[k+'_sd'])
    C = np.stack([zc(age, 'age'), np.nan_to_num(gender), np.nan_to_num(u1), np.nan_to_num(u2),
                  (np.isnan(u1) & np.isnan(u2)).astype(np.float32), zc(hadm, 'hadm'), zc(icu, 'icu')], 1).astype(np.float32)
    return Z, M, C

def decide(p, sigma, E, P):
    grey = (p >= P['tau_low']) & (p < P['tau_high']); unreliable = (sigma >= P['sigma_thr']) | (E <= P['E_thr'])
    return np.where(grey & unreliable, 'REVIEW', np.where(p >= P['tau'], 'ALERT', 'MONITOR'))

def sigmoid(x): return 1 / (1 + np.exp(-np.clip(x, -30, 30)))
print('✓ step 3 of 9 finished')

In [ ]:
RGN = ReliabilityGatedGRUD(); RGN.load_state_dict(B['rgn_state_dict']); RGN = RGN.to(device).eval()
LGBS = [lgb.Booster(model_file=os.path.join(BUNDLE_DIR, f)) for f in B['lightgbm_files']]
print(f'Loaded RG-GRU-D + {len(LGBS)} LightGBM models')

NICE = {'HR': 'heart rate', 'O2Sat': 'SpO₂', 'Temp': 'temperature', 'SBP': 'systolic BP', 'MAP': 'mean arterial pressure', 'DBP': 'diastolic BP',
        'Resp': 'respiratory rate', 'EtCO2': 'end-tidal CO₂', 'BaseExcess': 'base excess', 'HCO3': 'bicarbonate', 'FiO2': 'FiO₂ (oxygen given)',
        'pH': 'blood pH', 'PaCO2': 'PaCO₂', 'SaO2': 'SaO₂', 'AST': 'AST (liver)', 'BUN': 'BUN (urea, kidney)', 'Alkalinephos': 'alkaline phosphatase',
        'Calcium': 'calcium', 'Chloride': 'chloride', 'Creatinine': 'creatinine (kidney)', 'Bilirubin_direct': 'direct bilirubin', 'Glucose': 'glucose',
        'Lactate': 'lactate', 'Magnesium': 'magnesium', 'Phosphate': 'phosphate', 'Potassium': 'potassium', 'Bilirubin_total': 'total bilirubin (liver)',
        'TroponinI': 'troponin I (heart)', 'Hct': 'haematocrit', 'Hgb': 'haemoglobin', 'PTT': 'PTT (clotting)', 'WBC': 'white blood cells',
        'Fibrinogen': 'fibrinogen', 'Platelets': 'platelets', 'Age': 'age', 'Gender': 'sex (1 = male)', 'Unit1': 'medical ICU', 'Unit2': 'surgical ICU',
        'HospAdmTime': 'hours between hospital and ICU admission', 'ICULOS': 'hours in ICU', 'shock_index': 'shock index (HR ÷ SBP)',
        'sirs': 'SIRS criteria met', 'qsofa': 'qSOFA-lite score', 'labs_seen': 'lab types measured so far'}
SUFFIX = [('_dt', 'hours since {} measured'), ('_mean6h', '{} (6-h average)'), ('_std6h', '{} (6-h variability)'), ('_range6h', '{} (6-h range)'),
          ('_trend6h', '{} (6-h change)'), ('_trend24h', '{} (24-h change)'), ('_max', 'highest {} so far'), ('_min', 'lowest {} so far')]
def nice_name(c):
    for suf, tpl in SUFFIX:
        if c.endswith(suf): b = c[:-len(suf)]; return tpl.format(NICE.get(b, b))
    return NICE.get(c, c)
def nice_feat(c, v):
    v = float(v)
    if c.endswith('_dt') and (np.isnan(v) or v >= 998): return f"{NICE.get(c[:-3], c[:-3])}: never measured"
    if np.isnan(v): return f"{nice_name(c)}: not available"
    return f"{nice_name(c)} = {v:.3g}" if abs(v) < 1000 else f"{nice_name(c)} = {v:,.0f}"

class SepsisMonitor:
    """Feed one hour of raw bedside data at a time; returns risk, uncertainty, evidence, trust, the action and why."""
    def __init__(self): self.reset()
    def reset(self): self.X, self.D = [], []
    @torch.no_grad()
    def update(self, row):
        self.X.append([row.get(v, np.nan) for v in DYN]); self.D.append([row.get(v, np.nan) for v in DEMO])
        Xr, Dm = np.array(self.X, np.float32), np.array(self.D, np.float32); n = len(Xr)
        Z, M, C = to_model_inputs(Xr, Dm, NORM)
        tt = lambda a: torch.from_numpy(np.ascontiguousarray(a, dtype=np.float32))[None].to(device)
        f = featurize(tt(Z), tt(M), tt(C)); z, aux = RGN.encode(f); zl = z[:, -1:]
        rg = RGN.head_fwd(zl).item()
        rg_mc = torch.stack([RGN.head_fwd(zl, mc=True) for _ in range(MC)]).flatten().cpu().numpy()
        Fh = lgb_feats_from(Xr, ~np.isnan(Xr), np.zeros(n, int), np.arange(n), Dm).iloc[[-1]]
        lg = np.array([m.predict(Fh, raw_score=True)[0] for m in LGBS])
        b0, b1, b2 = W_STACK; K = len(lg)
        p = float(sigmoid(b0 + b1 * lg.mean() + b2 * rg))
        samp = sigmoid(np.array([b0 + b1 * lg[j % K] + b2 * rg_mc[j] for j in range(MC)]))
        E = float(f['E'][0, -1]); g = aux['gates'][0, -1].cpu().numpy()
        sig = float(samp.std())
        a = decide(np.array([p]), np.array([sig]), np.array([E]), POLICY)[0]
        lab_rows = np.flatnonzero((~np.isnan(Xr[:, NV:])).any(1))
        out = {'risk': p, 'lo': min(p, float(np.percentile(samp, 5))), 'hi': max(p, float(np.percentile(samp, 95))),
               'E': E, 'tv': float(g[0]), 'tl': float(g[1]), 'tc': float(g[2]), 'action': str(a), 'sigma': sig,
               'since_lab': int(n - 1 - lab_rows[-1]) if len(lab_rows) else None,
               'miss_key': [v for v in KEY_LABS if np.isnan(Xr[:, DYN.index(v)]).all()],
               'unc_high': sig >= POLICY['sigma_thr'], 'ev_low': E <= POLICY['E_thr'], 'drivers': []}
        if out['action'] == 'ALERT':            # exact TreeSHAP for this hour: the top measurements pushing risk up
            sh = np.mean([m.predict(Fh, pred_contrib=True)[0, :-1] for m in LGBS], 0)
            out['drivers'] = [nice_feat(Fh.columns[i], Fh.iloc[0, i]) for i in np.argsort(-sh)[:3] if sh[i] > 0]
        return out

def explain(Xr, Dm, t):
    """Exact TreeSHAP contributions (log-odds) of the LightGBM ensemble at hour t."""
    n = len(Xr); Fp = lgb_feats_from(Xr, ~np.isnan(Xr), np.zeros(n, int), np.arange(n), Dm).iloc[[t]]
    sh = np.mean([m.predict(Fp, pred_contrib=True)[0, :-1] for m in LGBS], 0)
    return Fp, sh
print('✓ step 4 of 9 finished')

In [ ]:
# ---- held-out TEST patients (never seen in training) ----
PSV = {}
for dp, dns, fns in os.walk(INPUT_ROOT, followlinks=True):
    dns[:] = [d for d in dns if not d.lower().startswith('images')]
    for fn in fns:
        if fn.endswith('.psv'): PSV.setdefault(fn[:-4], os.path.join(dp, fn))
test_ids = SPLIT_IDS['test']
_missing_test_ids = [p for p in test_ids if p not in PSV]
if _missing_test_ids:
    raise FileNotFoundError(f'{len(_missing_test_ids)} held-out patient files listed in split_patient_ids.json are missing under SEPSIS_INPUT_ROOT.')
if not test_ids:
    raise RuntimeError('split_patient_ids.json contains no held-out test patient IDs.')
rng = random.Random(42); sample = test_ids[:]; rng.shuffle(sample); sample = sample[:900]
def read_patient(path):
    df = pd.read_csv(path, sep='|' if path.endswith('.psv') else ',')
    for c in DYN + DEMO + ['SepsisLabel']:
        if c not in df.columns: df[c] = np.nan
    return df
septic, normal = [], []
for pid in sample:
    df = pd.read_csv(PSV[pid], sep='|', usecols=['SepsisLabel']); y = df.SepsisLabel.values; n = len(y)
    if y.max() == 1:
        onset = int(np.argmax(y)) + 6
        if onset >= 20 and n >= onset + 3 and n <= 200: septic.append((pid, n, onset))
    elif 24 <= n <= 150: normal.append((pid, n, None))
septic = sorted(septic, key=lambda r: r[2])[:25]; normal = normal[:12]
CHOICES = {f'{pid} · septic · sepsis onset at hour {o} · stay {n} h': (pid, o) for pid, n, o in septic}
CHOICES.update({f'{pid} · non-septic · stay {n} h': (pid, None) for pid, n, o in normal})
print(f'{len(septic)} septic + {len(normal)} non-septic held-out test patients available in the demo')
print('✓ step 5 of 9 finished')

In [ ]:
# Summary values are read from result CSVs in the selected bundle folder in the About cell.
# ---- plotting helpers ----
COL = {'risk': '#2a78d6', 'band': '#2a78d6', 'ALERT': '#d03b3b', 'REVIEW': '#f2b701', 'MONITOR': '#2e9e4f'}
def timeline_fig(log, n_total, onset, outage, req_hours, title):
    fig = Figure(figsize=(10, 5.4), dpi=100)
    ax, ax2, ax3 = fig.subplots(3, 1, sharex=True, gridspec_kw={'height_ratios': [2.4, 0.35, 1.1]})
    h = np.array([r['hour'] for r in log]); risk = np.array([r['risk'] for r in log]) * 100
    lo = np.array([r['lo'] for r in log]) * 100; hi = np.array([r['hi'] for r in log]) * 100
    ymax = max(5.0, float(hi.max()) * 1.15 if len(hi) else 5.0, POLICY['tau'] * 130)
    if outage is not None: [a.axvspan(outage - 0.5, n_total - 0.5, color='#8c8c8c', alpha=0.12, lw=0) for a in (ax, ax2, ax3)]
    ax.fill_between(h, lo, hi, color=COL['band'], alpha=0.18, lw=0, label='uncertainty band (5–95%)')
    ax.plot(h, risk, color=COL['risk'], lw=2.2, label='sepsis risk')
    ax.axhline(POLICY['tau'] * 100, color='#555', ls=':', lw=1.2, label=f"alert threshold τ = {POLICY['tau']*100:.1f}%")
    if onset is not None:
        for a in (ax, ax2, ax3): a.axvline(onset, color='#7a3fb0', lw=1.6)
        ax.text(onset + 0.4, ymax * 0.92, 'sepsis onset', color='#7a3fb0', fontsize=9)
    for rh in req_hours:
        for a in (ax, ax2, ax3): a.axvline(rh, color=COL['REVIEW'], ls='--', lw=1.4)
        ax.text(rh + 0.4, ymax * 0.80, 'REVIEW → labs requested', color='#a07800', fontsize=9)
    ax.set_xlim(-0.5, n_total - 0.5); ax.set_ylim(0, ymax); ax.set_ylabel('risk (%)'); ax.legend(loc='upper left', fontsize=8, frameon=False)
    ax.set_title(title, fontsize=11, loc='left'); ax.grid(alpha=0.25)
    ax2.bar(h, np.ones(len(h)), width=1.0, color=[COL[r['action']] for r in log]); ax2.set_yticks([]); ax2.set_ylabel('action', rotation=0, ha='right', va='center', fontsize=9)
    for k, c, lab in [('tv', '#2a78d6', 'trust: vitals'), ('tl', '#eb6834', 'trust: labs'), ('tc', '#1baf7a', 'trust: context')]:
        ax3.plot(h, [r[k] for r in log], color=c, lw=1.6, label=lab)
    ax3.plot(h, [r['E'] for r in log], color='#333', ls='--', lw=1.2, label='evidence E')
    ax3.set_ylim(-0.02, 1.05); ax3.set_xlabel('hours since ICU admission'); ax3.legend(loc='lower left', fontsize=7.5, ncol=4, frameon=False); ax3.grid(alpha=0.25)
    for a in (ax, ax2, ax3):
        a.spines['top'].set_visible(False); a.spines['right'].set_visible(False)
    fig.tight_layout(); return fig

def reason_short(r):
    rs = []
    if r['since_lab'] is None: rs.append('no lab results yet')
    elif r['since_lab'] >= 12: rs.append(f"last lab result {r['since_lab']} h ago")
    if r['ev_low']: rs.append(f"low evidence (E = {r['E']:.2f})")
    if r['unc_high']: rs.append('high model uncertainty')
    return rs or ['evidence below the reliability cut-off']

def why_html(r):
    p, tau = 100 * r['risk'], 100 * POLICY['tau']
    if r['action'] == 'REVIEW':
        mk = ', '.join(NICE.get(v, v).split(' (')[0] for v in r['miss_key'][:5])
        return (f"<b>Why REVIEW:</b> risk {p:.1f}% is in the grey zone ({tau/2:.1f}–{4*tau:.1f}%) and the evidence is weak: "
                + '; '.join(reason_short(r)) + '.' + (f"<br><b>Key labs never measured for this patient:</b> {mk}." if mk else '')
                + "<br><b>Suggested action:</b> obtain labs / bedside reassessment; the system re-decides every hour as evidence arrives.")
    if r['action'] == 'ALERT':
        lead = (f"risk {p:.1f}% is more than 4× the threshold ({tau:.1f}%), so it alerts even if evidence is incomplete" if r['risk'] >= POLICY['tau_high']
                else f"risk {p:.1f}% ≥ threshold {tau:.1f}% and the evidence is adequate")
        return f"<b>Why ALERT:</b> {lead}." + (f"<br><b>Top drivers now:</b> {'; '.join(r['drivers'])}." if r['drivers'] else '') + "<br><b>Suggested action:</b> notify the clinician for sepsis assessment."
    if r['risk'] < POLICY['tau_low']: return f"<b>Why MONITOR:</b> risk {p:.1f}% is low (below {tau/2:.1f}%)."
    return f"<b>Why MONITOR:</b> risk {p:.1f}% is below the threshold {tau:.1f}% and the evidence is adequate."

def status_html(r, labs_hidden, n_total):
    c = COL[r['action']]
    note = why_html(r)
    bar = lambda v, col: f'<div style="background:#e9edf0;border-radius:4px;height:8px;width:100%"><div style="width:{max(0,min(1,v))*100:.0f}%;height:8px;border-radius:4px;background:{col}"></div></div>'
    return f"""<div style="font-family:system-ui,sans-serif;display:flex;flex-direction:column;gap:10px">
      <div style="font-size:13px;opacity:.75">Hour <b>{r['hour']}</b> (stay: {n_total} h) · labs {'<b style="color:#a33">UNAVAILABLE</b>' if labs_hidden else 'available'}</div>
      <div><div style="font-size:12px;opacity:.75">SEPSIS RISK</div><div style="font-size:44px;font-weight:700;line-height:1">{r['risk']*100:.1f}%</div>
           <div style="font-size:13px;opacity:.75">uncertainty band {r['lo']*100:.1f}–{r['hi']*100:.1f}%</div></div>
      <div style="display:inline-block;border:2px solid {c};color:{c};font-weight:700;font-size:20px;padding:4px 14px;border-radius:8px;width:max-content">{r['action']}</div>
      <div style="font-size:13px;line-height:1.45;border-left:3px solid {c};padding-left:8px">{note}</div>
      <div style="display:grid;grid-template-columns:90px 1fr 40px;gap:6px;align-items:center;font-size:12.5px">
        <span>Evidence E</span>{bar(r['E'],'#333')}<span>{r['E']:.2f}</span>
        <span>Vitals</span>{bar(r['tv'],'#2a78d6')}<span>{r['tv']:.2f}</span>
        <span>Labs</span>{bar(r['tl'],'#eb6834')}<span>{r['tl']:.2f}</span>
        <span>Context</span>{bar(r['tc'],'#1baf7a')}<span>{r['tc']:.2f}</span></div></div>"""

def shap_fig(Fp, sh, t):
    top = np.argsort(-np.abs(sh))[:10][::-1]
    fig = Figure(figsize=(7.5, 3.6), dpi=100); ax = fig.subplots()
    ax.barh([nice_feat(Fp.columns[i], Fp.iloc[0, i]) for i in top], sh[top], color=['#d03b3b' if v > 0 else '#2a78d6' for v in sh[top]])
    ax.axvline(0, color='k', lw=0.8); ax.set_xlabel('contribution to risk (log-odds) · red raises risk')
    ax.set_title(f'Why the system flagged this patient at hour {t}', fontsize=11, loc='left')
    ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False); fig.tight_layout(); return fig
print('✓ step 6 of 9 finished')

In [ ]:
# ---- live run (a generator: the page updates every hour) ----
MON = SepsisMonitor()
def run_patient(choice, upload, outage_on, outage_start, seek_on, delay):
    if upload is not None:
        path = upload if isinstance(upload, str) else upload.name
        df = read_patient(path); onset = None
        if df['SepsisLabel'].fillna(0).max() == 1: onset = int(np.argmax(df['SepsisLabel'].fillna(0).values)) + 6
        title = f'Uploaded patient ({os.path.basename(path)})'
    else:
        if not choice: raise ValueError('Pick a patient first.')
        pid, onset = CHOICES[choice]; df = read_patient(PSV[pid]); title = f'Held-out test patient {pid}'
    X = df[DYN].values.astype(np.float32); D = df[DEMO].values.astype(np.float32); n = len(X)
    out_h = int(outage_start) if outage_on else None
    if out_h is not None and out_h >= n - 2: out_h = max(1, n // 2)   # outage set later than this stay: start it half-way
    MON.reset(); log, req, back, lat = [], [], None, []
    for t in range(n):
        row = dict(zip(DYN + DEMO, np.r_[X[t], D[t]]))
        hidden = outage_on and t >= out_h and (back is None or t < back)
        if hidden: row.update({v: np.nan for v in LABS})
        t1 = time.perf_counter(); r = MON.update(row); lat.append(time.perf_counter() - t1); r['hour'] = t
        if seek_on and hidden and r['action'] == 'REVIEW' and back is None: back = t + 1; req.append(t)
        log.append(r)
        yield timeline_fig(log, n, onset, out_h, req, title), status_html(r, hidden, n), None, f'Running… hour {t+1}/{n}'
        if delay > 0: time.sleep(delay)
    acts = np.array([r['action'] for r in log]); al = np.flatnonzero(acts == 'ALERT')
    lines = [f'**Average update time:** {1000*np.mean(lat):.1f} ms per hour (LightGBM ×{len(LGBS)} + GRU-D with {MC} uncertainty passes, {device})']
    if len(al):
        t_al = int(al[0])
        if onset is None:
            msg = (f'**First ALERT at hour {t_al}** · this patient never developed sepsis, so this is a **false alarm**. '
                   f'Across all {N_TEST:,} test patients, {FA_PCT:.0f}% of non-septic patients get at least one alert at this setting '
                   f'({FA_LB_PCT:.0f}% at the low-burden setting).')
        elif t_al <= onset: msg = f'**First ALERT at hour {t_al}** · sepsis onset at hour {onset} → **{onset - t_al} h early warning**'
        else: msg = f'**First ALERT at hour {t_al}** · sepsis onset at hour {onset} → alert came **{t_al - onset} h late**'
    else:
        t_al = int(np.argmax([r['risk'] for r in log]))
        msg = ('**No ALERT raised** · non-septic patient: correctly quiet' if onset is None else
               f'**No ALERT raised** · sepsis onset at hour {onset}: a **missed case**. Overall the system detects {DET_PCT:.0f}% of septic test patients.')
    lines.insert(0, msg)
    eps, i = [], 0
    while i < n:                                            # REVIEW episodes and what ended them
        if acts[i] == 'REVIEW':
            j = i
            while j + 1 < n and acts[j + 1] == 'REVIEW': j += 1
            nxt = acts[j + 1] if j + 1 < n else 'end of stay'
            eps.append(f"hours {i}–{j} ({', '.join(reason_short(log[i]))}) → {nxt}" if j > i else f"hour {i} ({', '.join(reason_short(log[i]))}) → {nxt}")
            i = j + 1
        else: i += 1
    if eps: lines.append(f'**REVIEW** in {100*np.mean(acts == "REVIEW"):.0f}% of hours · ' + '; '.join(eps[:4]) + (' …' if len(eps) > 4 else ''))
    if out_h is not None: lines.append(f'**Lab outage** simulated from hour {out_h}.')
    if req: lines.append(f'**Evidence-seeking:** REVIEW at hour {req[0]} requested labs; labs returned from hour {req[0]+1}.')
    Xh = X.copy()
    if outage_on:
        Xh[out_h:(back if back is not None else n), NV:] = np.nan
    Fp, sh = explain(Xh, D, t_al)
    yield timeline_fig(log, n, onset, out_h, req, title), status_html(log[-1], outage_on and (back is None) and n - 1 >= out_h, n), shap_fig(Fp, sh, t_al), '\n\n'.join(lines)
print('✓ step 7 of 9 finished')

In [ ]:
# ---- chest X-ray branch (separately trained DenseNet-121) ----
NIH = ['Atelectasis', 'Cardiomegaly', 'Effusion', 'Infiltration', 'Mass', 'Nodule', 'Pneumonia', 'Pneumothorax',
       'Consolidation', 'Edema', 'Emphysema', 'Fibrosis', 'Pleural_Thickening', 'Hernia']
TEST_AUROC = {}
for fn in ['cxr_auroc_test.csv', 'cxr_auroc_ci_test.csv']:
    fp = os.path.join(BUNDLE_DIR, fn)
    if not os.path.isfile(fp): continue
    try:
        d_ = pd.read_csv(fp, index_col=0)
        if 'AUROC' not in d_.columns or not d_.index.is_unique: continue
        values = {}
        for finding, value in d_['AUROC'].items():
            if finding not in NIH: continue
            try: auc = float(value)
            except (TypeError, ValueError): continue
            if np.isfinite(auc) and 0.0 <= auc <= 1.0: values[finding] = auc
        if values:
            TEST_AUROC = values
            break
    except Exception:
        continue
CXR_COUNT = len(TEST_AUROC)
CXR_MEAN = float(np.mean(list(TEST_AUROC.values()))) if CXR_COUNT else None
CNN = None
CXR_CHECKPOINT = os.path.join(BUNDLE_DIR, 'cxr_densenet121.pt')
if os.path.isfile(CXR_CHECKPOINT):
    try:
        from torchvision import models, transforms as T
        from PIL import Image
        CNN = models.densenet121(weights=None); CNN.classifier = nn.Linear(1024, len(NIH))
        CNN.load_state_dict(torch.load(CXR_CHECKPOINT, map_location='cpu')); CNN = CNN.to(device).eval()
        TF = T.Compose([T.Resize((224, 224)), T.ToTensor(), T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])])
        print('X-ray model loaded from', CXR_CHECKPOINT)
    except Exception as e:
        CNN = None; print('WARNING: X-ray model could not be loaded (the ICU demo still works):', repr(e))
else:
    print('X-ray model not found beside the selected bundle: the X-ray tab will not run inference.')

NIH_TEST = []
if CNN is not None and 'Data_Entry_2017.csv' in F_:
  try:
    from sklearn.model_selection import GroupShuffleSplit
    paths = {}
    for dp, _, fns in os.walk(os.path.dirname(F_['Data_Entry_2017.csv']), followlinks=True):
        for f in fns:
            if f.endswith('.png'): paths[f] = os.path.join(dp, f)
    meta = pd.read_csv(F_['Data_Entry_2017.csv']); meta = meta[meta['Image Index'].isin(paths)].reset_index(drop=True)
    _, gi_te = next(GroupShuffleSplit(1, test_size=0.15, random_state=42).split(meta, groups=meta['Patient ID']))   # same held-out patients as training
    te = meta.iloc[gi_te]; te = te[te['Finding Labels'].str.contains('Cardiomegaly|Effusion|Pneumothorax|Edema')]
    NIH_TEST = [(paths[r['Image Index']], r['Finding Labels']) for _, r in te.sample(min(300, len(te)), random_state=0).iterrows()]
    print(f'{len(NIH_TEST)} held-out NIH X-rays available for the "random test X-ray" button')
  except Exception as e:
    NIH_TEST = []; print('WARNING: NIH test images could not be indexed (upload your own X-ray instead):', repr(e))

def xray_predict(img, finding):
    if CNN is None: return None, None, 'X-ray model unavailable: attach a compatible checkpoint beside the selected bundle.'
    if img is None: return None, None, 'Upload a chest X-ray or press **Random held-out X-ray**.'
    img = img.convert('RGB'); x = TF(img)[None].to(device); store = {}
    def fh(m, i, o): store['a'] = o; o.register_hook(lambda g: store.__setitem__('g', g))
    hk = CNN.features.denseblock4.register_forward_hook(fh)
    with torch.enable_grad():
        CNN.zero_grad(); out = CNN(x); p = torch.sigmoid(out[0]).detach().cpu().numpy()
        k = NIH.index(finding) if finding in NIH else int(np.argmax(p)); out[0, k].backward()
    hk.remove()
    w = store['g'].mean((2, 3), keepdim=True); cam = F.relu((w * store['a']).sum(1, keepdim=True))
    cam = F.interpolate(cam, size=(224, 224), mode='bilinear', align_corners=False)[0, 0].detach().cpu().numpy(); cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
    f1 = Figure(figsize=(4.6, 4.6), dpi=100); a = f1.subplots(); a.imshow(img.convert('L').resize((224, 224)), cmap='gray'); a.imshow(cam, cmap='jet', alpha=0.4); a.axis('off')
    a.set_title(f'Grad-CAM: {NIH[k]} (p = {p[k]:.2f})', fontsize=10); f1.tight_layout()
    order = np.argsort(p); f2 = Figure(figsize=(5.4, 4.6), dpi=100); b = f2.subplots()
    b.barh([f'{NIH[i]}  (test AUROC {TEST_AUROC[NIH[i]]:.2f})' if NIH[i] in TEST_AUROC else f'{NIH[i]}  (AUROC unavailable)' for i in order], p[order], color=['#d03b3b' if i == k else '#8c98a4' for i in order])
    b.set_xlim(0, 1); b.set_xlabel('predicted probability'); b.set_title('14 findings', fontsize=10, loc='left')
    b.spines['top'].set_visible(False); b.spines['right'].set_visible(False); f2.tight_layout()
    metric_text = ('Results unavailable — run X-ray evaluation first.' if CXR_MEAN is None else
                   f'mean test AUROC {CXR_MEAN:.3f} over {CXR_COUNT} findings with valid CSV metrics')
    return f1, f2, f'Imaging branch, trained and evaluated separately on NIH ChestX-ray14 ({metric_text}). It shows thoracic findings, **not** sepsis.'

def random_xray():
    if not NIH_TEST: return None, 'NIH dataset not attached.'
    path, lab = random.choice(NIH_TEST)
    from PIL import Image
    return Image.open(path), f'True labels from the radiology report: **{lab}**'
print('✓ step 8 of 9 finished')

In [ ]:
_need = {'step 2 (load models)': 'BUNDLE_DIR', 'step 4 (monitor)': 'SepsisMonitor', 'step 5 (test patients)': 'CHOICES',
         'step 6 (plots)': 'timeline_fig', 'step 7 (live run)': 'run_patient'}
_miss = [k for k, v in _need.items() if v not in globals()]
if _miss:
    raise RuntimeError('These earlier steps have not run yet: ' + ', '.join(_miss) +
                       '. Use Run All from the top (or Run → Restart & Run All) and wait for every cell to print "✓ step … finished".')
if 'xray_predict' not in globals():   # X-ray step skipped: the ICU monitor still works
    NIH = ['Atelectasis', 'Cardiomegaly', 'Effusion', 'Infiltration', 'Mass', 'Nodule', 'Pneumonia', 'Pneumothorax',
           'Consolidation', 'Edema', 'Emphysema', 'Fibrosis', 'Pleural_Thickening', 'Hernia']
    CXR_MEAN = None
    CXR_COUNT = 0
    def xray_predict(img, finding): return None, None, 'Results unavailable — run X-ray evaluation first.'
    def random_xray(): return None, 'X-ray step (step 8) did not run.'
import gradio as gr
import re
def _r(fn, **kw):
    fp = os.path.join(BUNDLE_DIR, fn); return pd.read_csv(fp, **kw) if os.path.exists(fp) else None
def _diff(cell):          # '+0.054 [+0.03, +0.07] (p < 0.005)'  ->  '+0.054 (p < 0.005)'
    m = re.match(r'\s*([+-]?\d+\.\d+)\s*\[.*\]\s*\((p[^)]*)\)', str(cell))
    if not m or not np.isfinite(float(m.group(1))): raise ValueError('missing or invalid paired-test result')
    return f'{m.group(1)} ({m.group(2)})'
UNAVAILABLE = 'Results unavailable — required result file or valid row missing'
def _num(value):
    value = float(value)
    if not np.isfinite(value): raise ValueError('result value is not finite')
    return value
K = {key: UNAVAILABLE for key in ['auc', 'miss_u', 'ext', 'rec', 'conf', 'abl', 'rev', 'xc']}
try:
    R = _r('main_results_test.csv', index_col=0); h = R.loc[[i_ for i_ in R.index if i_.startswith('Hybrid')][0]]
    _sirs = R.loc[[i_ for i_ in R.index if i_.startswith('SIRS')][0], 'AUROC']; _qsofa = R.loc[[i_ for i_ in R.index if i_.startswith('qSOFA')][0], 'AUROC']
    K['auc'] = f"**{_num(h.AUROC):.3f}** [{_num(h.AUROC_lo):.3f}–{_num(h.AUROC_hi):.3f}] (SIRS rule {_num(_sirs):.3f}, qSOFA {_num(_qsofa):.3f})"
except Exception: pass
try:
    PT = _r('v4_paired_tests.csv'); hy = PT[PT.comparison.str.startswith('Hybrid') & PT.comparison.str.endswith('LightGBM ensemble')]
    for key, ts in [('miss_u', 'internal, labs_missing'), ('ext', 'external hospital B (trained on A)')]:
        r_ = hy[hy['test set'] == ts].iloc[0]
        K[key] = f"utility **{_diff(r_['Δ utility [95% CI] (p)'])}**, AUROC {_diff(r_['Δ AUROC [95% CI] (p)'])}"
except Exception: pass
try:
    EV = _r('evidence_seeking_experiment.csv', index_col=0)['utility (threshold)'].values
    _ev = [_num(v) for v in EV[:3]]
    if len(_ev) < 3 or _ev[0] == _ev[1]: raise ValueError('invalid evidence-seeking result row')
    K['rec'] = f"utility {_ev[1]:.3f} → **{_ev[2]:.3f}** (all labs: {_ev[0]:.3f}), recovers **{100*(_ev[2]-_ev[1])/(_ev[0]-_ev[1]):.0f}%** of the loss"
except Exception: pass
try:
    CF = _r('v4_conformal_review.csv', index_col=0); r_ = CF.loc[min(CF.index, key=lambda a_: abs(a_ - 0.1))]
    K['conf'] = f"{_num(r_['sepsis hours dismissed as MONITOR %']):.1f}% of sepsis hours missed on unseen patients, REVIEW in {_num(r_['REVIEW hours %']):.1f}% of hours"
except Exception: pass
try:
    AB_ = _r('v4_shortcut_ablation.csv', index_col=0); K['abl'] = f"AUROC {_num(AB_.loc['Hybrid without time + measurement pattern', 'AUROC']):.3f}"
except Exception: pass
ABOUT = f"""
### What this system does
Every hour, it reads an ICU patient's vitals, labs and context, estimates **sepsis risk with an uncertainty band**,
and chooses an action: **ALERT** (act now), **MONITOR** (continue) or **REVIEW** (risk not negligible but evidence too weak: obtain labs / reassess).

### Key results (PhysioNet/CinC 2019 · held-out test patients · read from this run's result files)
| | Result |
|---|---|
| Hybrid risk engine, test AUROC | {K['auc']} |
| When all labs are missing (vs LightGBM) | {K['miss_u']} |
| Unseen hospital (train A → test B, vs LightGBM) | {K['ext']} |
| Lab outage: REVIEW requests labs | {K['rec']} |
| Conformal REVIEW (target miss rate 10%) | {K['conf']} |
| How often REVIEW fires (all evidence) | {K['rev']} |
| Without time-in-ICU and test-ordering features | {K['abl']} |
| Chest X-ray branch (separate task, NIH) | {('Results unavailable — run X-ray evaluation first.' if CXR_MEAN is None else f'mean AUROC {CXR_MEAN:.3f} over {CXR_COUNT} findings with valid CSV metrics')} |
| Unrelated X-rays added to ICU patients (control) | {K['xc']} |

*Research prototype for clinician decision support, not a diagnostic device. Imaging is not fused with the ICU data
because no public dataset pairs them for the same patients without credentialed access (MIMIC-IV + MIMIC-CXR: next step).*
"""
HOWTO = f"""
* **Blue line:** calibrated sepsis risk ("5%" ≈ 5 in 100 such hours). **Shaded band:** model uncertainty (5–95%).
* **Dotted line:** alert threshold τ = {POLICY['tau']*100:.1f}%, chosen on validation patients to maximise the official PhysioNet clinical-utility score.
  Baseline event rates depend on the attached cohort and are not asserted by this demo.
* **Action bar:** green **MONITOR**, amber **REVIEW** (risk in the grey zone {POLICY['tau_low']*100:.1f}–{POLICY['tau_high']*100:.1f}% *and* weak evidence or high
  uncertainty → obtain labs / reassess), red **ALERT**. Each hour's reason is shown in the left panel.
* **Bottom panel:** evidence score E (½ freshness of vitals + ½ coverage of labs) and the internal trust weight per data stream.
  Trust weights are model internals, **not** explanations (our noise-control experiment showed this).
* **Purple line:** sepsis onset from the dataset label. **Grey shading:** simulated lab outage. **Amber dashed line:** REVIEW requested labs.
* **Red/blue bars (bottom chart):** exact TreeSHAP, how much each measurement pushed the risk up (red) or down (blue) at the first ALERT.
"""
with gr.Blocks(title='Sepsis Early-Warning Demo') as demo:
    gr.Markdown('# Sepsis Early-Warning System · live demo\nReliability-aware clinical decision support · runs the trained models in real time · **research prototype, not for diagnosis**')
    with gr.Tab('Live ICU monitor'):
        with gr.Row():
            with gr.Column(scale=1, min_width=300):
                choice = gr.Dropdown(list(CHOICES), label='Held-out test patient (never seen in training)', value=list(CHOICES)[0] if CHOICES else None)
                upload = gr.File(label='…or upload a patient file (.psv / .csv, PhysioNet columns)', file_types=['.psv', '.csv'])
                outage = gr.Checkbox(label='Simulate a lab outage', value=False)
                out_start = gr.Slider(0, 200, value=24, step=1, label='Lab outage starts at hour')
                seek = gr.Checkbox(label='Evidence-seeking: REVIEW requests labs', value=True)
                delay = gr.Slider(0, 0.5, value=0.05, step=0.05, label='Playback delay per hour (s)')
                go = gr.Button('▶ Run live', variant='primary')
                status = gr.HTML()
            with gr.Column(scale=3):
                plot = gr.Plot(show_label=False)
                summary = gr.Markdown()
                why = gr.Plot(show_label=False)
                with gr.Accordion('How to read this screen', open=False):
                    gr.Markdown(HOWTO)
        go.click(run_patient, [choice, upload, outage, out_start, seek, delay], [plot, status, why, summary])
    with gr.Tab('Chest X-ray branch'):
        with gr.Row():
            with gr.Column(scale=1, min_width=300):
                img = gr.Image(type='pil', label='Chest X-ray')
                rnd = gr.Button('Random held-out X-ray')
                lab = gr.Markdown()
                fsel = gr.Dropdown(['(top finding)'] + NIH, value='(top finding)', label='Grad-CAM for')
                xb = gr.Button('Analyse X-ray', variant='primary')
            with gr.Column(scale=2):
                with gr.Row():
                    cam = gr.Plot(show_label=False); bars = gr.Plot(show_label=False)
                xnote = gr.Markdown()
        rnd.click(random_xray, None, [img, lab])
        xb.click(xray_predict, [img, fsel], [cam, bars, xnote])
    with gr.Tab('About & results'):
        gr.Markdown(ABOUT)

SHARE = os.environ.get('DEMO_SHARE', '1') == '1'
BLOCK = os.environ.get('DEMO_BLOCK', '0') == '1'
demo.queue().launch(share=SHARE, server_port=int(os.environ.get('DEMO_PORT', 7860)), prevent_thread_lock=not BLOCK, show_error=True)